# Instant Gratification

## Your first Frontier LLM Project!

Let's build a useful LLM solution - in a matter of minutes.

By the end of this course, you will have built an autonomous Agentic AI solution with 7 agents that collaborate to solve a business problem. All in good time! We will start with something smaller...

Our goal is to code a new kind of Web Browser. Give it a URL, and it will respond with a summary. The Reader's Digest of the internet!!

Before starting, you should have completed the setup for [PC](../SETUP-PC.md) or [Mac](../SETUP-mac.md) and you hopefully launched this jupyter lab from within the project root directory, with your environment activated.

## If you're new to Jupyter Lab

Welcome to the wonderful world of Data Science experimentation! Once you've used Jupyter Lab, you'll wonder how you ever lived without it. Simply click in each "cell" with code in it, such as the cell immediately below this text, and hit Shift+Return to execute that cell. As you wish, you can add a cell with the + button in the toolbar, and print values of variables, or try out variations.  

I've written a notebook called [Guide to Jupyter](Guide%20to%20Jupyter.ipynb) to help you get more familiar with Jupyter Labs, including adding Markdown comments, using `!` to run shell commands, and `tqdm` to show progress.

If you prefer to work in IDEs like VSCode or Pycharm, they both work great with these lab notebooks too.  


In [1]:
# imports

import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

# Connecting to OpenAI

The next cell is where we load in the environment variables in your `.env` file and connect to OpenAI.

## Troubleshooting if you have problems:

Head over to the [troubleshooting](troubleshooting.ipynb) notebook in this folder for step by step code to identify the root cause and fix it!

If you make a change, try restarting the "Kernel" (the python process sitting behind this notebook) by Kernel menu >> Restart Kernel and Clear Outputs of All Cells. Then try this notebook again, starting at the top.

Or, contact me! Message me or email ed@edwarddonner.com and we will get this to work.

Any concerns about API costs? See my notes in the README - costs should be minimal, and you can control it at every point. You can also use Ollama as a free alternative, which we discuss during Day 2.

In [2]:
# Load environment variables in a file called .env

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [3]:
openai = OpenAI()

# If this doesn't work, try Kernel menu >> Restart Kernel and Clear Outputs Of All Cells, then run the cells from the top of this notebook down.
# If it STILL doesn't work (horrors!) then please see the troubleshooting notebook, or try the below line instead:
# openai = OpenAI(api_key="your-key-here-starting-sk-proj-")

# Let's make a quick call to a Frontier model to get started, as a preview!

In [4]:
# To give you a preview -- calling OpenAI with these messages is this easy:

message = "Hello, GPT! This is my first ever message to you! Hi!"
response = openai.chat.completions.create(model="gpt-4o-mini", messages=[{"role":"user", "content":message}])
print(response.choices[0].message.content)

Hello! Welcome! I'm glad you reached out. How can I assist you today?


## OK onwards with our first project

In [5]:
# A class to represent a Webpage
# If you're not familiar with Classes, check out the "Intermediate Python" notebook

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [6]:
# Let's try one out. Change the website and add print statements to follow along.

ed = Website("https://edwarddonner.com")
print(ed.title)
print(ed.text)

Home - Edward Donner
Home
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy DJing (but I’m badly out of practice), amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. Recruiters use our product today to source, understand, engage and manage talent. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
We work with groundbreaking, proprietary LLMs verticalized for talent, we’ve
patented
our matching model, and our award-winning platform has happy customers and tons of press coverage.
Connec

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT4o have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [7]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

In [8]:
# A function that writes a User Prompt that asks for summaries of websites:

def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

In [9]:
print(user_prompt_for(ed))

You are looking at a website titled Home - Edward Donner
The contents of this website is as follows; please provide a short summary of this website in markdown. If it includes news or announcements, then summarize these too.

Home
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy DJing (but I’m badly out of practice), amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. Recruiters use our product today to source, understand, engage and manage talent. I’m previously the founder and CEO of AI startup untapt,
acqui

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]

To give you a preview, the next 2 cells make a rather simple call - we won't stretch the might GPT (yet!)

In [10]:
messages = [
    {"role": "system", "content": "You are a snarky assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

In [11]:
# To give you a preview -- calling OpenAI with system and user messages:

response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
print(response.choices[0].message.content)

Oh, a tough one! Are you sure you’re ready for the answer? It's 4! Mind-blowing, right?


## And now let's build useful messages for GPT-4o-mini, using a function

In [12]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [13]:
# Try this out, and then try for a few more websites

messages_for(ed)

[{'role': 'system',
  'content': 'You are an assistant that analyzes the contents of a website and provides a short summary, ignoring text that might be navigation related. Respond in markdown.'},
 {'role': 'user',
  'content': 'You are looking at a website titled Home - Edward Donner\nThe contents of this website is as follows; please provide a short summary of this website in markdown. If it includes news or announcements, then summarize these too.\n\nHome\nConnect Four\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy DJing (but I’m badly out of practice), amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of\nNebula.io\n. We’re applying AI to a field where it can make a

## Time to bring it together - the API for OpenAI is very simple!

In [14]:
# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model = "gpt-4o-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [15]:
summarize("https://edwarddonner.com")

"# Summary of Edward Donner's Website\n\nEdward Donner's website serves as a personal platform where he shares his interests and professional background. He is the co-founder and CTO of **Nebula.io**, which focuses on applying AI to help individuals discover their potential and engage with talent. Previously, he founded **untapt**, an AI startup acquired in 2021. \n\nIn addition to his expertise in coding and experimenting with large language models (LLMs), he enjoys DJing and electronic music production.\n\n## Recent News and Announcements:\n- **January 23, 2025**: Resources from the LLM Workshop – Hands-on with Agents.\n- **December 21, 2024**: Welcoming SuperDataScientists.\n- **November 13, 2024**: Resources for Mastering AI and LLM Engineering.\n- **October 16, 2024**: Resources for transitioning from Software Engineer to AI Data Scientist. \n\nEdward encourages connections with like-minded individuals interested in these fields."

In [16]:
# A function to display this nicely in the Jupyter output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [17]:
display_summary("https://edwarddonner.com")

# Summary of Edward Donner's Website

Edward Donner’s website serves as a platform for sharing his interests in coding, experiments with Large Language Models (LLMs), and electronic music. As the co-founder and CTO of Nebula.io, he focuses on utilizing AI to help individuals discover their potential, particularly in the talent recruitment sector. Previously, he founded an AI startup named untapt, which was acquired in 2021.

## Recent News and Announcements

- **January 23, 2025**: LLM Workshop – Hands-on with Agents – resources are available.
- **December 21, 2024**: Welcome message for SuperDataScientists.
- **November 13, 2024**: Resources for "Mastering AI and LLM Engineering" are provided.
- **October 16, 2024**: Resources available for transitioning from Software Engineer to AI Data Scientist. 

Overall, the site emphasizes Edward's expertise in AI, especially in the context of talent management, and offers various resources and announcements related to LLM workshops and education.

# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.

Websites that are rendered with Javascript, like React apps, won't show up. See the community-contributions folder for a Selenium implementation that gets around this. You'll need to read up on installing Selenium (ask ChatGPT!)

Also Websites protected with CloudFront (and similar) may give 403 errors - many thanks Andy J for pointing this out.

But many websites will work just fine!

In [18]:
display_summary("https://cnn.com")

# CNN Website Summary

CNN is a leading news outlet providing breaking news and in-depth coverage across a variety of topics including U.S. and world news, politics, business, health, entertainment, sports, and science. The site offers a curated selection of videos, articles, and live updates, ensuring users stay informed on major global events.

### Recent Highlights:
- **Russia-Ukraine Conflict**: A recent warning about Russia’s military actions after the U.S. curtailed military aid has raised concerns among European leaders.
- **Trump Administration**: Trump threatens new tariffs on Canada, including a 250% tax on dairy products, highlighting economic tensions.
- **Global Events**: Reports on various topics such as South Carolina's first execution by firing squad in 15 years, the ongoing Israel-Hamas conflict, and the economic impacts of the Biden administration’s policies.
- **Science**: A cache of 1.5 million-year-old tools was uncovered by archaeologists, shedding light on ancient societies.

### Features:
- CNN offers multiple formats for consuming news, including live TV, podcasts, and newsletters tailored to diverse interests.
- The site also engages users with quizzes and interactive content to enhance the news experience.

Overall, CNN maintains its reputation for delivering timely and relevant news to its audience.

In [19]:
display_summary("https://anthropic.com")

# Anthropic Website Summary

Anthropic is a research and AI safety company based in San Francisco, focusing on developing reliable and beneficial AI systems. The main features of the website include:

- **Claude AI Models**: The latest release, **Claude 3.7 Sonnet**, is highlighted as the company's most advanced AI model, which integrates hybrid reasoning capabilities. There is also a tool named **Claude Code**, aimed at facilitating coding tasks.

- **Research and Safety**: The company emphasizes the importance of AI safety, with published research insights and commitments to ethical practices and transparency in AI development.

## Recent Announcements:
- **Claude 3.7 Sonnet** and **Claude Code** were launched, marking significant advancements in AI technology.

- **Alignment Research**: Ongoing efforts to refine AI safety strategies, including the announcement titled "Constitutional AI: Harmlessness from AI Feedback," which was released on December 15, 2022.

- Discussions on AI safety methodology were shared in the announcement "Core Views on AI Safety: When, Why, What, and How" on March 8, 2023.

## Additional Features:
- The website offers various resources, including documentation for developers, pricing plans for different users, and career opportunities at Anthropic.

## An extra exercise for those who enjoy web scraping

You may notice that if you try `display_summary("https://openai.com")` - it doesn't work! That's because OpenAI has a fancy website that uses Javascript. There are many ways around this that some of you might be familiar with. For example, Selenium is a hugely popular framework that runs a browser behind the scenes, renders the page, and allows you to query it. If you have experience with Selenium, Playwright or similar, then feel free to improve the Website class to use them. In the community-contributions folder, you'll find an example Selenium solution from a student (thank you!)

# Sharing your code

I'd love it if you share your code afterwards so I can share it with others! You'll notice that some students have already made changes (including a Selenium implementation) which you will find in the community-contributions folder. If you'd like add your changes to that folder, submit a Pull Request with your new versions in that folder and I'll merge your changes.

If you're not an expert with git (and I am not!) then GPT has given some nice instructions on how to submit a Pull Request. It's a bit of an involved process, but once you've done it once it's pretty clear. As a pro-tip: it's best if you clear the outputs of your Jupyter notebooks (Edit >> Clean outputs of all cells, and then Save) for clean notebooks.

PR instructions courtesy of an AI friend: https://chatgpt.com/share/670145d5-e8a8-8012-8f93-39ee4e248b4c

# **Web Scraping for JavaScript Website**

In [20]:
# !pip install selenium
# !pip install undetected-chromedriver

In [21]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
from bs4 import BeautifulSoup

In [22]:
class WebsiteCrawler:
    def __init__(self, url, wait_time=20, chrome_binary_path=None):
        """
        Initialize the WebsiteCrawler using Selenium to scrape JavaScript-rendered content.
        """
        self.url = url
        self.wait_time = wait_time

        options = uc.ChromeOptions()
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--disable-blink-features=AutomationControlled")
        options.add_argument("start-maximized")
        options.add_argument(
            "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
        )
        if chrome_binary_path:
            options.binary_location = chrome_binary_path

        self.driver = uc.Chrome(options=options)

        try:
            # Load the URL
            self.driver.get(url)

            # Wait for Cloudflare or similar checks
            time.sleep(10)

            # Ensure the main content is loaded
            WebDriverWait(self.driver, self.wait_time).until(
                EC.presence_of_element_located((By.TAG_NAME, "main"))
            )

            # Extract the main content
            main_content = self.driver.find_element(By.CSS_SELECTOR, "main").get_attribute("outerHTML")

            # Parse with BeautifulSoup
            soup = BeautifulSoup(main_content, "html.parser")
            self.title = self.driver.title if self.driver.title else "No title found"
            self.text = soup.get_text(separator="\n", strip=True)

        except Exception as e:
            print(f"Error occurred: {e}")
            self.title = "Error occurred"
            self.text = ""

        finally:
            self.driver.quit()


In [23]:
# chrome_path = "C:/Program Files/Google/Chrome/Application/chrome.exe"
chrome_path = "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"
url = "https://www.canva.com/"


In [24]:
def new_summary(url, chrome_path):
    web = WebsiteCrawler(url, 30, chrome_path)
    response = openai.chat.completions.create(
            model = "gpt-4o-mini",
            messages = messages_for(web)
        )

    web_summary = response.choices[0].message.content
    
    return display(Markdown(web_summary))

In [25]:
new_summary(url, chrome_path)

# Canva: Visual Suite for Everyone

Canva is a versatile online design platform that enables users to easily create professional-grade designs and share or print them. It offers a range of design options, including posters, resumes, logos, presentations, social media graphics, and more.

## Key Features

- **Design Types:** Canva provides templates for various creative projects, including videos, brochures, Instagram posts, and websites.
- **Pricing Tiers:** Users can choose from different plans: 
  - **Canva Free:** Basic features for individuals.
  - **Canva Pro:** Enhanced features for individual brand growth.
  - **Canva Teams:** Collaboration tools for groups.
  - **Canva Enterprise:** Customized solutions for organizations.
  - Special offerings for educational institutions and nonprofits featuring premium access at no cost.

## Collaboration and AI Tools

- **Real-Time Collaboration:** Invite others to co-design and provide feedback through comments.
- **AI Features:** Utilize tools like Magic Write for copy generation and Magic Edit for photo enhancements.

## Printing Services

Canva also allows users to design and print products such as business cards, flyers, and photo albums, with the convenience of free delivery.

## Sustainability Commitment

Canva emphasizes sustainability by planting a tree for every printing service used and maintaining carbon-neutral operations.

## Noteworthy Quotes from Users

- Many businesses report enhanced efficiency and cost savings due to Canva's intuitive design platform, enabling even non-designers to produce quality content.

Overall, Canva is designed to meet the needs of individuals, teams, and organizations looking for an easy and effective design solution, all while prioritizing sustainability and collaboration.

In [26]:
url = "https://openai.com"

In [27]:
new_summary(url, chrome_path)

# OpenAI Website Summary

The OpenAI website serves as a platform for engaging with ChatGPT, offering various functionalities such as helping users with writing, planning, coding, translating, and much more. Users can input specific requests to receive tailored responses, enhancing their productivity across numerous tasks.

## Recent News and Announcements
1. **Introducing GPT‑4.5** (Feb 27, 2025): An overview of the latest iteration of OpenAI's language model.
2. **Deep Research Launch** (Feb 2, 2025): Announcement of a new feature focusing on in-depth research capabilities.
3. **Safety Model Specification** (Feb 12, 2025): Details on the most recent advancements in safety measures for AI applications.
4. **AI in Education** (Feb 10, 2025): Collaboration with the CSU system to integrate AI resources for 500,000 students and faculty.
5. **ChatGPT Gov Introduction** (Jan 31, 2025): Launch of a dedicated platform for governmental interactions using ChatGPT.
6. **The Stargate Project** (Jan 21, 2025): Announcement of a unique initiative aimed at enhancing AI functionality and services.
7. **Partnership with Axios** (Jan 15, 2025): Expansion of OpenAI’s efforts in working with the news industry to enhance information delivery systems.

The site focuses on providing tools and updates surrounding OpenAI's technologies, particularly through the use of AI to assist in various real-world applications.